In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr


DATA_DIR = "../../../data/shop"


# Load required data.
orders = pd.read_csv(
    f"{DATA_DIR}/orders.csv",
    usecols=["order_id", "user_id", "eval_set"],
)

order_products = pd.read_csv(
    f"{DATA_DIR}/order_products__prior.csv",
    usecols=[
        "order_id",
        "product_id",
        "reordered",
    ],
)

products = pd.read_csv(
    f"{DATA_DIR}/products.csv",
    usecols=["product_id", "aisle_id"],
)


# Add user and aisle information to prior purchases.
prior_orders = orders[
    orders["eval_set"].eq("prior")
][["order_id", "user_id"]]

purchases = (
    order_products
    .merge(
        prior_orders,
        on="order_id",
        how="inner",
    )
    .merge(
        products,
        on="product_id",
        how="left",
    )
    .dropna(subset=["aisle_id"])
)


# Calculate user-level measures.
user_metrics = (
    purchases
    .groupby("user_id")
    .agg(
        unique_aisle_count=(
            "aisle_id",
            "nunique",
        ),
        reorder_rate=(
            "reordered",
            "mean",
        ),
        total_prior_orders=(
            "order_id",
            "nunique",
        ),
    )
    .reset_index()
)


# Initial correlation.
initial_r, initial_p = pearsonr(
    user_metrics["unique_aisle_count"],
    user_metrics["reorder_rate"],
)


# Remove the linear effect of total prior orders.
def residualize(y, control):
    return y - np.polyval(
        np.polyfit(control, y, 1),
        control,
    )


aisle_residual = residualize(
    user_metrics["unique_aisle_count"],
    user_metrics["total_prior_orders"],
)

reorder_residual = residualize(
    user_metrics["reorder_rate"],
    user_metrics["total_prior_orders"],
)


# Partial correlation.
partial_r, partial_p = pearsonr(
    aisle_residual,
    reorder_residual,
)


print(
    "Initial correlation:",
    f"{initial_r:.3f}",
)

print(
    "Partial correlation controlling for "
    "total prior orders:",
    f"{partial_r:.3f}",
)

Initial correlation: 0.314
Partial correlation controlling for total prior orders: -0.084
